In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the dataset
dataset_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single'

# Check if the path exists and list the first few files
if os.path.exists(dataset_path):
    print(f"Successfully accessed: {dataset_path}")
    files = os.listdir(dataset_path)
    print(f"Total items found: {len(files)}")
    print("Sample files:", files[:5])
else:
    print(f"Path not found: {dataset_path}. Please ensure the folder name is correct.")

Mounted at /content/drive
Successfully accessed: /content/drive/MyDrive/Colab Notebooks/MVSA_Single
Total items found: 9
Sample files: ['data', 'labelResultAll.txt', 'mvsa_single_processed.parquet', 'checkpoints', 'vilbert_final']


In [2]:
import pandas as pd

import os

from PIL import Image

import io

import gc

from concurrent.futures import ThreadPoolExecutor



# Paths

labels_file = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/labelResultAll.txt'

data_dir = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/data'

parquet_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed_224.parquet'

target_image_size = (224, 224)



def process_single_row(row):

    """Process one image-text pair and store the image at the final training size."""

    try:

        item_id = str(int(row['id']))

        image_path = os.path.join(data_dir, f"{item_id}.jpg")

        text_path = os.path.join(data_dir, f"{item_id}.txt")



        t_label = str(row['text']).strip()

        i_label = str(row['image']).strip()



        if t_label == 'neutral':

            f_label = i_label

        elif i_label == 'neutral':

            f_label = t_label

        elif t_label == i_label:

            f_label = t_label

        else:

            return None



        if os.path.exists(image_path) and os.path.exists(text_path):

            with Image.open(image_path) as img:

                img = img.convert('RGB')

                img = img.resize(target_image_size, Image.Resampling.LANCZOS)

                buf = io.BytesIO()

                img.save(buf, format='PNG')

                img_bytes = buf.getvalue()



            with open(text_path, 'r', encoding='latin-1') as file:

                text_content = file.read().strip()



            return {

                'ID': item_id,

                'text_sentiment': t_label,

                'image_sentiment': i_label,

                'final_sentiment': f_label,

                'image_bytes': img_bytes,

                'text_content': text_content

            }

    except Exception as e:

        print(f"Error processing row: {e}")

        return None

    return None



if os.path.exists(parquet_path):

    print(f"File parquet sudah ada, memuat dari: {parquet_path}")

    df = pd.read_parquet(parquet_path)

    print(f"Total data dimuat: {len(df)}")

    print("\nDistribusi sentimen:")

    print(df['final_sentiment'].value_counts())

    display(df.head())

else:

    print('File parquet tidak ditemukan, memulai pemrosesan...')

    print(f"Membaca labels dari: {labels_file}")



    df_labels = pd.read_csv(labels_file, sep=',', header=0)

    print(f"Total labels dibaca: {len(df_labels)}")

    print(f"Nama kolom dalam file labels: {list(df_labels.columns)}")

    print(f"Sample data:\n{df_labels.head()}")



    print(f"\nMemulai pemrosesan paralel dengan resizing gambar ke {target_image_size[0]}x{target_image_size[1]}...")

    rows = [row for _, row in df_labels.iterrows()]



    with ThreadPoolExecutor(max_workers=4) as executor:

        results = list(executor.map(process_single_row, rows))



    processed_records = [r for r in results if r is not None]

    print(f"Total records berhasil diproses: {len(processed_records)}")



    if processed_records:

        df = pd.DataFrame(processed_records)

        print(f"Menyimpan data ke: {parquet_path}")

        df.to_parquet(parquet_path, index=False)

        print('Data berhasil disimpan!')



        del results, processed_records, df_labels, rows

        gc.collect()



        print(f"\nTotal data berhasil diproses dan disimpan: {len(df)}")

        print("\nDistribusi sentimen:")

        print(df['final_sentiment'].value_counts())

        display(df.head())

    else:

        print('PERINGATAN: Tidak ada data yang berhasil diproses!')

        print('Periksa apakah:')

        print(f"  1. File label ada di: {labels_file}")

        print(f"  2. Folder data ada di: {data_dir}")

        print('  3. File .jpg dan .txt ada di dalam folder data')


File parquet sudah ada, memuat dari: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total data dimuat: 4511

Distribusi sentimen:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."


In [3]:
import pandas as pd



parquet_save_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed_224.parquet'



try:

    df.to_parquet(parquet_save_path, index=False)

    print(f"Dataset berhasil disimpan ke: {parquet_save_path}")

    print(f"Ukuran file: {os.path.getsize(parquet_save_path) / (1024 * 1024):.2f} MB")

except Exception as e:

    print(f"Terjadi kesalahan saat menyimpan: {e}")


Dataset berhasil disimpan ke: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Ukuran file: 853.77 MB


In [4]:
# ─────────────────────────────────────────────

# Cell 3 — Install & Import Dependencies

# ─────────────────────────────────────────────

%pip install transformers accelerate scikit-learn pyarrow torchvision timm -q



import copy

import io

import json

import math

import os

import random

import warnings



import numpy as np

import pandas as pd

from PIL import Image, ImageFile



import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.cuda.amp import GradScaler, autocast

from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW



import timm

import torchvision.transforms as transforms



from transformers import BertModel, BertTokenizer, get_cosine_schedule_with_warmup

from sklearn.metrics import (

    accuracy_score,

    classification_report,

    confusion_matrix,

    f1_score,

    precision_recall_fscore_support,

)

from sklearn.model_selection import train_test_split



warnings.filterwarnings('ignore')

ImageFile.LOAD_TRUNCATED_IMAGES = True



print('All libraries imported successfully.')

print(f"PyTorch version : {torch.__version__}")

print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"GPU             : {torch.cuda.get_device_name(0)}")

    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


All libraries imported successfully.
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU Memory      : 15.6 GB


In [5]:
# ─────────────────────────────────────────────

# Cell 4 — Model Configuration

# ─────────────────────────────────────────────

CONFIG = {

    # ── Paths ──────────────────────────────────────────

    "parquet_path": "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed_224.parquet",

    "checkpoint_dir": "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_v2_checkpoints",

    "save_dir": "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_v2_final",



    # ── Text Encoder ───────────────────────────────────

    "bert_model_name": "bert-base-uncased",

    "bert_hidden_size": 768,

    "max_text_length": 128,



    # ── Image Encoder ──────────────────────────────────

    "image_backbone": "vit_base_patch16_224",

    "image_feature_dim": 768,

    "image_proj_dim": 768,

    "image_input_size": 224,

    "image_drop_cls_token": True,

    "image_trainable_blocks": 4,

    "image_dropout": 0.10,



    # ── Co-Attention ───────────────────────────────────

    "co_attn_layers": 2,

    "co_attn_heads": 8,

    "co_attn_dropout": 0.10,

    "ffn_hidden_dim": 3072,



    # ── Fusion & Classifier ────────────────────────────

    "fusion_strategy": "cls-image-absdiff-gated",

    "fusion_input_dim": 768 * 3,

    "classifier_hidden": 768,

    "num_classes": 3,

    "classifier_dropout": 0.25,



    # ── Training ───────────────────────────────────────

    "epochs": 20,

    "batch_size": 16,

    "grad_accum_steps": 2,

    "bert_lr": 1.5e-5,

    "vision_lr": 5e-5,

    "new_params_lr": 1.5e-4,

    "weight_decay": 5e-4,

    "warmup_ratio": 0.10,

    "gradient_clip": 1.0,

    "early_stop_patience": 5,

    "label_smoothing": 0.05,

    "loss_name": "weighted_ce",

    "use_amp": True,



    # ── Data Split ─────────────────────────────────────

    "train_ratio": 0.80,

    "val_ratio": 0.10,

    "test_ratio": 0.10,



    # ── Reproducibility ────────────────────────────────

    "seed": 42,

    "num_workers": 2,

    "device": "cuda" if torch.cuda.is_available() else "cpu",

}



def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)



set_seed(CONFIG["seed"])

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

os.makedirs(CONFIG["save_dir"], exist_ok=True)



SEP = "=" * 60

print(SEP)

print("  ViLBERT v2 Multimodal Sentiment Analysis — Configuration")

print(SEP)

print(f"  Device                 : {CONFIG['device'].upper()}")

if torch.cuda.is_available():

    print(f"  GPU                    : {torch.cuda.get_device_name(0)}")



print("\n  ── Text Encoder ──────────────────────────────────────")

print(f"  Model                  : {CONFIG['bert_model_name']}")

print(f"  Hidden size            : {CONFIG['bert_hidden_size']}")

print(f"  Max token length       : {CONFIG['max_text_length']}")



print("\n  ── Image Encoder ─────────────────────────────────────")

print(f"  Backbone               : {CONFIG['image_backbone']}")

print(f"  Feature dim            : {CONFIG['image_feature_dim']}")

print(f"  Drop CLS token         : {CONFIG['image_drop_cls_token']}")

print(f"  Trainable ViT blocks   : {CONFIG['image_trainable_blocks']}")

print(f"  Input image size       : {CONFIG['image_input_size']}x{CONFIG['image_input_size']}")



print("\n  ── Co-Attention ──────────────────────────────────────")

print(f"  Layers                 : {CONFIG['co_attn_layers']}")

print(f"  Heads                  : {CONFIG['co_attn_heads']}")

print(f"  FFN hidden dim         : {CONFIG['ffn_hidden_dim']}")

print(f"  Dropout                : {CONFIG['co_attn_dropout']}")



print("\n  ── Fusion & Classifier ───────────────────────────────")

print(f"  Fusion strategy        : {CONFIG['fusion_strategy']}")

print(f"  Fusion input dim       : {CONFIG['fusion_input_dim']}")

print(f"  Classifier hidden      : {CONFIG['classifier_hidden']}")

print(f"  Classifier dropout     : {CONFIG['classifier_dropout']}")



print("\n  ── Training Hyperparameters ──────────────────────────")

print(f"  Epochs                 : {CONFIG['epochs']}")

print(f"  Batch size             : {CONFIG['batch_size']}")

print(f"  Grad accum steps       : {CONFIG['grad_accum_steps']}")

print(f"  BERT LR                : {CONFIG['bert_lr']}")

print(f"  Vision LR              : {CONFIG['vision_lr']}")

print(f"  New-params LR          : {CONFIG['new_params_lr']}")

print(f"  Weight decay           : {CONFIG['weight_decay']}")

print(f"  Warmup ratio           : {CONFIG['warmup_ratio']} ({int(CONFIG['warmup_ratio'] * 100)}% of steps)")

print(f"  Label smoothing        : {CONFIG['label_smoothing']}")

print(f"  Loss                   : {CONFIG['loss_name']}")

print(f"  Automatic mixed precision: {CONFIG['use_amp']}")



print("\n  ── Data ──────────────────────────────────────────────")

print(f"  Split (train/val/test) : {CONFIG['train_ratio']}/{CONFIG['val_ratio']}/{CONFIG['test_ratio']}")

print(f"  Random seed            : {CONFIG['seed']}")

print(SEP)


  ViLBERT Multimodal Sentiment Analysis — Configuration
  Device              : CUDA
  GPU                 : Tesla T4

  ── Text Encoder ──────────────────────────────────────
  Model               : bert-base-uncased
  Hidden size         : 768
  BERT layers         : 12
  BERT attention heads: 12
  Max token length    : 128

  ── Image Encoder ─────────────────────────────────────
  Backbone            : resnet101
  Feature dim (raw)   : 2048
  Projected dim       : 768
  Input image size    : 224×224

  ── Co-Attention Mechanism ────────────────────────────
  Type                : multi-head (bidirectional)
  Number of layers    : 2
  Number of heads     : 8
  FFN hidden dim      : 3072
  Dropout             : 0.1

  ── Fusion & Classifier ───────────────────────────────
  Fusion strategy     : concatenation
  Fusion input dim    : 1536
  Classifier hidden   : 512
  Num output classes  : 3  (neg / neu / pos)
  Classifier dropout  : 0.3

  ── Training Hyperparameters ────────────────

In [6]:
# ─────────────────────────────────────────────
# Cell 5 — Load Parquet & Prepare Data Splits
# ─────────────────────────────────────────────
print(f"Loading parquet from: {CONFIG['parquet_path']}")
df_full = pd.read_parquet(CONFIG["parquet_path"])
print(f"Total rows loaded : {len(df_full)}")
print(f"Columns           : {list(df_full.columns)}")

# ── Drop any NaN rows ────────────────────────────────────────────────
before = len(df_full)
df_full = df_full.dropna(subset=["text_content", "image_bytes", "final_sentiment"]).reset_index(drop=True)
print(f"Rows after dropna : {len(df_full)}  (dropped {before - len(df_full)})")

# ── Encode labels: negative→0, neutral→1, positive→2 ────────────────
LABEL_MAP  = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_IMAP = {v: k for k, v in LABEL_MAP.items()}

df_full["label"] = df_full["final_sentiment"].str.strip().map(LABEL_MAP)
missing_labels = df_full["label"].isna().sum()
if missing_labels > 0:
    print(f"WARN: {missing_labels} rows have unrecognised labels — dropping them.")
    df_full = df_full.dropna(subset=["label"]).reset_index(drop=True)
df_full["label"] = df_full["label"].astype(int)

print(f"\nLabel distribution (final_sentiment):")
for lbl, idx in LABEL_MAP.items():
    count = (df_full["label"] == idx).sum()
    pct   = count / len(df_full) * 100
    print(f"  {idx}  {lbl:<10}  {count:>5}  ({pct:.1f}%)")

# ── Stratified split: 80 / 10 / 10 ─────────────────────────────────
df_train, df_tmp = train_test_split(
    df_full,
    test_size=(CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_full["label"],
    random_state=CONFIG["seed"],
)
df_val, df_test = train_test_split(
    df_tmp,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_tmp["label"],
    random_state=CONFIG["seed"],
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"\nData split (stratified):")
print(f"  Train : {len(df_train):>5} rows")
print(f"  Val   : {len(df_val):>5} rows")
print(f"  Test  : {len(df_test):>5} rows")
print(f"  Total : {len(df_train)+len(df_val)+len(df_test):>5} rows")


Loading parquet from: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total rows loaded : 4511
Columns           : ['ID', 'text_sentiment', 'image_sentiment', 'final_sentiment', 'image_bytes', 'text_content']
Rows after dropna : 4511  (dropped 0)

Label distribution (final_sentiment):
  0  negative     1358  (30.1%)
  1  neutral       470  (10.4%)
  2  positive     2683  (59.5%)

Data split (stratified):
  Train :  3608 rows
  Val   :   451 rows
  Test  :   452 rows
  Total :  4511 rows


In [7]:
# ─────────────────────────────────────────────

# Cell 6 — Dataset Class & DataLoaders

# ─────────────────────────────────────────────

tokenizer = BertTokenizer.from_pretrained(CONFIG["bert_model_name"])



IMAGE_MEAN = [0.485, 0.456, 0.406]

IMAGE_STD = [0.229, 0.224, 0.225]



train_transform = transforms.Compose([

    transforms.RandomResizedCrop(

        CONFIG["image_input_size"],

        scale=(0.88, 1.0),

        ratio=(0.9, 1.1),

    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomApply([

        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03)

    ], p=0.4),

    transforms.RandomAffine(degrees=6, translate=(0.03, 0.03)),

    transforms.ToTensor(),

    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),

])



eval_transform = transforms.Compose([

    transforms.Resize(256),

    transforms.CenterCrop(CONFIG["image_input_size"]),

    transforms.ToTensor(),

    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),

])





class MVSADataset(Dataset):

    def __init__(self, dataframe, tokenizer, image_transform, max_length):

        self.df = dataframe.reset_index(drop=True)

        self.tokenizer = tokenizer

        self.transform = image_transform

        self.max_len = max_length



    def __len__(self):

        return len(self.df)



    @staticmethod

    def _decode_image_bytes(img_bytes):

        if isinstance(img_bytes, bytes):

            raw_bytes = img_bytes

        elif isinstance(img_bytes, bytearray):

            raw_bytes = bytes(img_bytes)

        elif isinstance(img_bytes, memoryview):

            raw_bytes = img_bytes.tobytes()

        elif isinstance(img_bytes, np.ndarray):

            raw_bytes = img_bytes.tobytes()

        else:

            raw_bytes = bytes(img_bytes)

        return Image.open(io.BytesIO(raw_bytes)).convert("RGB")



    def __getitem__(self, idx):

        row = self.df.iloc[idx]



        try:

            image = self._decode_image_bytes(row["image_bytes"])

            corrupted = 0

        except Exception:

            image = Image.new("RGB", (CONFIG["image_input_size"], CONFIG["image_input_size"]), color=(0, 0, 0))

            corrupted = 1



        image_tensor = self.transform(image)



        encoding = self.tokenizer(

            str(row["text_content"]),

            padding="max_length",

            truncation=True,

            max_length=self.max_len,

            return_tensors="pt",

        )



        return {

            "input_ids": encoding["input_ids"].squeeze(0),

            "attention_mask": encoding["attention_mask"].squeeze(0),

            "token_type_ids": encoding["token_type_ids"].squeeze(0),

            "image": image_tensor,

            "label": torch.tensor(row["label"], dtype=torch.long),

            "corrupted": torch.tensor(corrupted, dtype=torch.long),

        }





loader_kwargs = {

    "batch_size": CONFIG["batch_size"],

    "num_workers": CONFIG["num_workers"],

    "pin_memory": torch.cuda.is_available(),

}

if CONFIG["num_workers"] > 0:

    loader_kwargs["persistent_workers"] = True



train_dataset = MVSADataset(df_train, tokenizer, train_transform, CONFIG["max_text_length"])

val_dataset = MVSADataset(df_val, tokenizer, eval_transform, CONFIG["max_text_length"])

test_dataset = MVSADataset(df_test, tokenizer, eval_transform, CONFIG["max_text_length"])



train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)

val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)



print(f"Train batches : {len(train_loader)}  ({len(train_dataset)} samples)")

print(f"Val   batches : {len(val_loader)}  ({len(val_dataset)} samples)")

print(f"Test  batches : {len(test_loader)}  ({len(test_dataset)} samples)")



batch = next(iter(train_loader))

print("\nBatch shapes:")

print(f"  input_ids      : {batch['input_ids'].shape}")

print(f"  attention_mask : {batch['attention_mask'].shape}")

print(f"  token_type_ids : {batch['token_type_ids'].shape}")

print(f"  image          : {batch['image'].shape}")

print(f"  label          : {batch['label'].shape}   values={batch['label'].tolist()}")

print(f"  corrupted      : {batch['corrupted'].sum().item()} sample(s) in this batch")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches : 226  (3608 samples)
Val   batches : 29  (451 samples)
Test  batches : 29  (452 samples)

Batch shapes:
  input_ids      : torch.Size([16, 128])
  attention_mask : torch.Size([16, 128])
  token_type_ids : torch.Size([16, 128])
  image          : torch.Size([16, 3, 224, 224])
  label          : torch.Size([16])   values=[2, 2, 2, 2, 0, 2, 0, 0, 2, 1, 2, 2, 0, 1, 0, 0]


In [8]:
# ─────────────────────────────────────────────

# Cell 7 — Co-Attention Mechanism

# ─────────────────────────────────────────────

class LearnedAttentionPool(nn.Module):

    def __init__(self, hidden_dim, dropout):

        super().__init__()

        self.score = nn.Sequential(

            nn.Linear(hidden_dim, hidden_dim),

            nn.Tanh(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1),

        )



    def forward(self, hidden_states, mask=None):

        scores = self.score(hidden_states).squeeze(-1)

        if mask is not None:

            scores = scores.masked_fill(~mask, -1e4)

        weights = torch.softmax(scores, dim=-1)

        pooled = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)

        return pooled





class CoAttentionLayer(nn.Module):

    def __init__(self, hidden_dim, num_heads, ffn_dim, dropout):

        super().__init__()

        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"



        self.t2v_attn = nn.MultiheadAttention(

            embed_dim=hidden_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True,

        )

        self.t2v_norm1 = nn.LayerNorm(hidden_dim)

        self.t2v_ffn = nn.Sequential(

            nn.Linear(hidden_dim, ffn_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(ffn_dim, hidden_dim),

            nn.Dropout(dropout),

        )

        self.t2v_norm2 = nn.LayerNorm(hidden_dim)



        self.v2t_attn = nn.MultiheadAttention(

            embed_dim=hidden_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True,

        )

        self.v2t_norm1 = nn.LayerNorm(hidden_dim)

        self.v2t_ffn = nn.Sequential(

            nn.Linear(hidden_dim, ffn_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(ffn_dim, hidden_dim),

            nn.Dropout(dropout),

        )

        self.v2t_norm2 = nn.LayerNorm(hidden_dim)



    def forward(self, text_feats, image_feats, text_key_padding_mask=None):

        if text_key_padding_mask is not None:

            all_masked = text_key_padding_mask.all(dim=1)

            if all_masked.any():

                text_key_padding_mask = text_key_padding_mask.clone()

                text_key_padding_mask[all_masked] = False



        t_attn_out, _ = self.t2v_attn(

            query=text_feats,

            key=image_feats,

            value=image_feats,

        )

        text_out = self.t2v_norm1(text_feats + t_attn_out)

        text_out = self.t2v_norm2(text_out + self.t2v_ffn(text_out))



        v_attn_out, _ = self.v2t_attn(

            query=image_feats,

            key=text_feats,

            value=text_feats,

            key_padding_mask=text_key_padding_mask,

        )

        image_out = self.v2t_norm1(image_feats + v_attn_out)

        image_out = self.v2t_norm2(image_out + self.v2t_ffn(image_out))

        return text_out, image_out





_B, _Lt, _Lv, _d = 2, 12, 196, 768

_layer = CoAttentionLayer(hidden_dim=_d, num_heads=8, ffn_dim=3072, dropout=0.1)

_pool = LearnedAttentionPool(hidden_dim=_d, dropout=0.1)

_t = torch.randn(_B, _Lt, _d)

_v = torch.randn(_B, _Lv, _d)

_m = torch.ones(_B, _Lt, dtype=torch.bool)

_to, _vo = _layer(_t, _v, ~_m)

_pooled = _pool(_vo)

print(f"CoAttentionLayer OK  |  text_out: {tuple(_to.shape)}  image_out: {tuple(_vo.shape)}")

print(f"Attention pooling OK  |  pooled image: {tuple(_pooled.shape)}")

del _layer, _pool, _t, _v, _m, _to, _vo, _pooled


CoAttentionLayer OK  |  text_out: (2, 12, 768)  image_out: (2, 49, 768)


In [9]:
# ─────────────────────────────────────────────

# Cell 8 — ViLBERT v2 Model Architecture

# ─────────────────────────────────────────────

class ViTImageEncoder(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.backbone = timm.create_model(

            config["image_backbone"],

            pretrained=True,

            num_classes=0,

            global_pool="",

        )

        self.drop_cls_token = config["image_drop_cls_token"]

        self.dropout = nn.Dropout(config["image_dropout"])

        self._freeze_backbone(config["image_trainable_blocks"])



    def _freeze_backbone(self, trainable_blocks):

        for param in self.backbone.parameters():

            param.requires_grad = False



        if hasattr(self.backbone, "blocks"):

            for block in self.backbone.blocks[-trainable_blocks:]:

                for param in block.parameters():

                    param.requires_grad = True



        for attr in ["norm", "fc_norm", "head_norm", "patch_embed"]:

            module = getattr(self.backbone, attr, None)

            if module is not None:

                for param in module.parameters():

                    param.requires_grad = True



        for attr in ["cls_token", "pos_embed"]:

            value = getattr(self.backbone, attr, None)

            if isinstance(value, nn.Parameter):

                value.requires_grad = True



    def forward(self, pixel_values):

        tokens = self.backbone.forward_features(pixel_values)

        if isinstance(tokens, (list, tuple)):

            tokens = tokens[-1]

        if tokens.dim() == 4:

            tokens = tokens.flatten(2).transpose(1, 2)

        if self.drop_cls_token and tokens.size(1) > 1:

            tokens = tokens[:, 1:, :]

        return self.dropout(tokens)





class ViLBERT(nn.Module):

    def __init__(self, config):

        super().__init__()

        hidden_dim = config["bert_hidden_size"]



        self.bert = BertModel.from_pretrained(config["bert_model_name"])

        self.image_encoder = ViTImageEncoder(config)

        self.text_norm = nn.LayerNorm(hidden_dim)

        self.image_norm = nn.LayerNorm(hidden_dim)



        self.co_attn_layers = nn.ModuleList([

            CoAttentionLayer(

                hidden_dim=hidden_dim,

                num_heads=config["co_attn_heads"],

                ffn_dim=config["ffn_hidden_dim"],

                dropout=config["co_attn_dropout"],

            )

            for _ in range(config["co_attn_layers"])

        ])



        self.image_pool = LearnedAttentionPool(hidden_dim, config["classifier_dropout"])

        self.fusion_gate = nn.Sequential(

            nn.Linear(config["fusion_input_dim"], hidden_dim),

            nn.GELU(),

            nn.Dropout(config["classifier_dropout"]),

            nn.Linear(hidden_dim, config["fusion_input_dim"]),

            nn.Sigmoid(),

        )

        self.classifier = nn.Sequential(

            nn.Linear(config["fusion_input_dim"], config["classifier_hidden"]),

            nn.LayerNorm(config["classifier_hidden"]),

            nn.GELU(),

            nn.Dropout(config["classifier_dropout"]),

            nn.Linear(config["classifier_hidden"], config["num_classes"]),

        )



    def forward(self, input_ids, attention_mask, token_type_ids, pixel_values):

        text_outputs = self.bert(

            input_ids=input_ids,

            attention_mask=attention_mask,

            token_type_ids=token_type_ids,

        )

        text_feats = self.text_norm(text_outputs.last_hidden_state)

        image_feats = self.image_norm(self.image_encoder(pixel_values))



        text_key_padding_mask = attention_mask == 0

        for layer in self.co_attn_layers:

            text_feats, image_feats = layer(

                text_feats,

                image_feats,

                text_key_padding_mask=text_key_padding_mask,

            )



        cls_token = text_feats[:, 0, :]

        image_context = self.image_pool(image_feats)

        fused = torch.cat([cls_token, image_context, torch.abs(cls_token - image_context)], dim=-1)

        fused = fused * self.fusion_gate(fused)

        logits = self.classifier(fused)

        return logits





device = torch.device(CONFIG["device"])

model = ViLBERT(CONFIG).to(device)



total_params = sum(param.numel() for param in model.parameters())

trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)

vision_trainable_params = sum(

    param.numel() for param in model.image_encoder.backbone.parameters() if param.requires_grad

)

print(f"Total parameters         : {total_params:,}")

print(f"Trainable parameters     : {trainable_params:,}")

print(f"Trainable vision params  : {vision_trainable_params:,}")



model.eval()

with torch.no_grad():

    _dummy = next(iter(train_loader))

    _logits = model(

        _dummy["input_ids"].to(device),

        _dummy["attention_mask"].to(device),

        _dummy["token_type_ids"].to(device),

        _dummy["image"].to(device),

    )

print(f"Forward pass OK  |  logits shape: {tuple(_logits.shape)}   (expected [B, 3])")

del _dummy, _logits

model.train()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 179MB/s] 


Total parameters    : 182,698,563
Trainable parameters: 182,698,563
Forward pass OK  |  logits shape: (16, 3)   (expected [B, 3])


ViLBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tr

In [10]:
# ─────────────────────────────────────────────

# Cell 9 — Training Loop

# ─────────────────────────────────────────────

from sklearn.utils.class_weight import compute_class_weight as _cwf





class WeightedFocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):

        super().__init__()

        self.alpha = alpha

        self.gamma = gamma

        self.reduction = reduction



    def forward(self, logits, targets):

        ce = F.cross_entropy(logits, targets, reduction="none")

        pt = torch.exp(-ce)

        focal = (1 - pt).pow(self.gamma) * ce

        if self.alpha is not None:

            focal = self.alpha[targets] * focal

        if self.reduction == "mean":

            return focal.mean()

        if self.reduction == "sum":

            return focal.sum()

        return focal





def build_loss(config, class_weights):

    if config["loss_name"] == "focal":

        return WeightedFocalLoss(alpha=class_weights, gamma=2.0)

    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config["label_smoothing"])





def move_batch_to_device(batch, device):

    return {

        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value

        for key, value in batch.items()

    }





def evaluate(loader, model, criterion, device):

    model.eval()

    total_loss = 0.0

    valid_batches = 0

    corrupted_samples = 0

    all_preds, all_labels = [], []



    with torch.no_grad():

        for batch in loader:

            batch = move_batch_to_device(batch, device)

            corrupted_samples += batch["corrupted"].sum().item()



            with autocast(enabled=CONFIG["use_amp"] and device.type == "cuda"):

                logits = model(

                    batch["input_ids"],

                    batch["attention_mask"],

                    batch["token_type_ids"],

                    batch["image"],

                )

                loss = criterion(logits, batch["label"])



            if not torch.isfinite(loss):

                continue



            total_loss += loss.item()

            valid_batches += 1

            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())

            all_labels.extend(batch["label"].cpu().tolist())



    avg_loss = total_loss / max(valid_batches, 1)

    acc = accuracy_score(all_labels, all_preds) if all_labels else 0.0

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) if all_labels else 0.0

    precision, recall, per_class_f1, support = precision_recall_fscore_support(

        all_labels,

        all_preds,

        labels=np.arange(CONFIG["num_classes"]),

        zero_division=0,

    ) if all_labels else (

        np.zeros(CONFIG["num_classes"]),

        np.zeros(CONFIG["num_classes"]),

        np.zeros(CONFIG["num_classes"]),

        np.zeros(CONFIG["num_classes"]),

    )



    return {

        "loss": avg_loss,

        "acc": acc,

        "f1": f1,

        "preds": all_preds,

        "labels": all_labels,

        "precision": precision,

        "recall": recall,

        "per_class_f1": per_class_f1,

        "support": support,

        "corrupted_samples": int(corrupted_samples),

    }





bert_param_ids = {id(param) for param in model.bert.parameters()}

vision_param_ids = {

    id(param) for param in model.image_encoder.backbone.parameters() if param.requires_grad

}



bert_params = [param for param in model.parameters() if id(param) in bert_param_ids]

vision_params = [param for param in model.parameters() if id(param) in vision_param_ids]

new_params = [

    param

    for param in model.parameters()

    if param.requires_grad and id(param) not in bert_param_ids and id(param) not in vision_param_ids

]



optimizer_groups = [

    {"params": bert_params, "lr": CONFIG["bert_lr"]},

    {"params": vision_params, "lr": CONFIG["vision_lr"]},

    {"params": new_params, "lr": CONFIG["new_params_lr"]},

]

optimizer = AdamW(optimizer_groups, weight_decay=CONFIG["weight_decay"])



updates_per_epoch = math.ceil(len(train_loader) / CONFIG["grad_accum_steps"])

total_steps = updates_per_epoch * CONFIG["epochs"]

warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

scheduler = get_cosine_schedule_with_warmup(

    optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_steps,

)



class_weights_np = _cwf(

    class_weight="balanced",

    classes=np.arange(CONFIG["num_classes"]),

    y=df_train["label"].values,

)

class_weights = torch.tensor(class_weights_np, dtype=torch.float, device=device)

criterion = build_loss(CONFIG, class_weights)

scaler = GradScaler(enabled=CONFIG["use_amp"] and device.type == "cuda")



print(

    f"Class weights  ─  negative={class_weights_np[0]:.3f}  "

    f"neutral={class_weights_np[1]:.3f}  positive={class_weights_np[2]:.3f}"

)

print(f"Optimizer groups ─ BERT={len(bert_params)}  Vision={len(vision_params)}  New={len(new_params)} tensors")



best_val_f1 = 0.0

best_ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "vilbert_v2_best.pt")

patience_counter = 0

history = []

experiment_rows = []



print(

    f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  "

    f"{'Val Acc':>7}  {'Val F1':>7}  {'LR(new)':>10}  {'Status'}"

)

print("-" * 94)



for epoch in range(1, CONFIG["epochs"] + 1):

    model.train()

    optimizer.zero_grad(set_to_none=True)



    train_loss = 0.0

    train_preds, train_labels = [], []

    nan_batches = 0

    corrupted_samples = 0



    for step, batch in enumerate(train_loader, start=1):

        batch = move_batch_to_device(batch, device)

        corrupted_samples += batch["corrupted"].sum().item()



        with autocast(enabled=CONFIG["use_amp"] and device.type == "cuda"):

            logits = model(

                batch["input_ids"],

                batch["attention_mask"],

                batch["token_type_ids"],

                batch["image"],

            )

            loss = criterion(logits, batch["label"])



        if not torch.isfinite(loss):

            nan_batches += 1

            optimizer.zero_grad(set_to_none=True)

            continue



        scaler.scale(loss / CONFIG["grad_accum_steps"]).backward()



        should_step = step % CONFIG["grad_accum_steps"] == 0 or step == len(train_loader)

        if should_step:

            scaler.unscale_(optimizer)

            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])

            scaler.step(optimizer)

            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            scheduler.step()



        train_loss += loss.item()

        train_preds.extend(logits.detach().argmax(dim=-1).cpu().tolist())

        train_labels.extend(batch["label"].cpu().tolist())



    train_loss = train_loss / max(len(train_loader) - nan_batches, 1)

    train_acc = accuracy_score(train_labels, train_preds) if train_labels else 0.0



    val_metrics = evaluate(val_loader, model, criterion, device)

    lr_new = optimizer.param_groups[-1]["lr"]



    status = f"patience {patience_counter}/{CONFIG['early_stop_patience']}"

    if val_metrics["f1"] > best_val_f1:

        best_val_f1 = val_metrics["f1"]

        patience_counter = 0

        status = "best saved"

        torch.save(

            {

                "epoch": epoch,

                "model_state": model.state_dict(),

                "optimizer_state": optimizer.state_dict(),

                "scheduler_state": scheduler.state_dict(),

                "val_metrics": val_metrics,

                "config": copy.deepcopy(CONFIG),

            },

            best_ckpt_path,

        )

    else:

        patience_counter += 1

        status = f"patience {patience_counter}/{CONFIG['early_stop_patience']}"

        if patience_counter >= CONFIG["early_stop_patience"]:

            print(

                f"\nEarly stopping at epoch {epoch}  "

                f"(no improvement for {CONFIG['early_stop_patience']} epochs)"

            )

            history.append({

                "epoch": epoch,

                "train_loss": train_loss,

                "train_acc": train_acc,

                "val_loss": val_metrics["loss"],

                "val_acc": val_metrics["acc"],

                "val_f1": val_metrics["f1"],

                "lr_new": lr_new,

                "nan_batches": nan_batches,

                "corrupted_train": int(corrupted_samples),

            })

            break



    history.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "train_acc": train_acc,

        "val_loss": val_metrics["loss"],

        "val_acc": val_metrics["acc"],

        "val_f1": val_metrics["f1"],

        "lr_new": lr_new,

        "nan_batches": nan_batches,

        "corrupted_train": int(corrupted_samples),

    })



    experiment_rows.append({

        "epoch": epoch,

        "val_f1": val_metrics["f1"],

        "val_acc": val_metrics["acc"],

        "val_loss": val_metrics["loss"],

        "best_so_far": best_val_f1,

    })



    print(

        f"{epoch:>5}  {train_loss:>10.4f}  {train_acc:>9.4f}  {val_metrics['loss']:>8.4f}  "

        f"{val_metrics['acc']:>7.4f}  {val_metrics['f1']:>7.4f}  {lr_new:>10.2e}  {status}"

    )



print(f"\nTraining complete. Best validation macro-F1: {best_val_f1:.4f}")

print(f"Best checkpoint path: {best_ckpt_path}")


Class weights  ─  negative=1.107  neutral=3.199  positive=0.560
Epoch  Train Loss  Train Acc  Val Loss  Val Acc   Val F1        Status
───────────────────────────────────────────────────────────────────────────
    1      1.0025     0.5335    0.9935   0.6962   0.5857  ✓ best saved
    2      0.7663     0.6929    0.8474   0.6541   0.5898  ✓ best saved
    3      0.5315     0.8043    1.0333   0.7162   0.6245  ✓ best saved
    4      0.2702     0.9060    1.4374   0.6741   0.6173              
    5      0.2390     0.9360    2.1716   0.7029   0.6118              
    6      0.1194     0.9734    2.4390   0.7140   0.6320  ✓ best saved
    7      0.1087     0.9803    3.0090   0.7140   0.6126              
    8      0.0675     0.9864    3.1459   0.6984   0.6046              
    9      0.0501     0.9889    3.4080   0.7184   0.6181              

Early stopping at epoch 10  (no improvement for 4 epochs)

Training complete.  Best val macro-F1: 0.6320
Best checkpoint saved to: /content/drive/MyD

In [11]:
# ─────────────────────────────────────────────

# Cell 10 — Evaluation on Test Set

# ─────────────────────────────────────────────

print(f"Loading best checkpoint: {best_ckpt_path}")

checkpoint = torch.load(best_ckpt_path, map_location=device)

model.load_state_dict(checkpoint["model_state"])

print(

    f"Checkpoint epoch: {checkpoint['epoch']}  "

    f"val_f1={checkpoint['val_metrics']['f1']:.4f}  "

    f"val_acc={checkpoint['val_metrics']['acc']:.4f}"

)



test_metrics = evaluate(test_loader, model, criterion, device)

class_names = [LABEL_IMAP[i] for i in range(CONFIG["num_classes"])]

cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])



SEP = "=" * 60

print(f"\n{SEP}")

print("  Test Set Results — ViLBERT v2")

print(SEP)

print(f"  Accuracy  (overall)        : {test_metrics['acc']:.4f}  ({test_metrics['acc'] * 100:.2f}%)")

print(f"  Macro F1  (unweighted)     : {test_metrics['f1']:.4f}")

print(f"  Corrupted samples seen     : {test_metrics['corrupted_samples']}")

print(f"  Best validation macro-F1   : {best_val_f1:.4f}")



print("\n  Per-Class Metrics:")

for idx, name in enumerate(class_names):

    print(

        f"  {name:<8}  precision={test_metrics['precision'][idx]:.4f}  "

        f"recall={test_metrics['recall'][idx]:.4f}  "

        f"f1={test_metrics['per_class_f1'][idx]:.4f}  "

        f"support={int(test_metrics['support'][idx])}"

    )



print("\n  Full Classification Report:")

print(

    classification_report(

        test_metrics["labels"],

        test_metrics["preds"],

        target_names=class_names,

        digits=4,

        zero_division=0,

    )

)



print("  Confusion Matrix  (rows=true, cols=pred):")

header = "          " + "  ".join(f"{name:>10}" for name in class_names)

print(header)

for idx, row in enumerate(cm):

    row_str = "  ".join(f"{value:>10}" for value in row)

    print(f"  {class_names[idx]:>8}  {row_str}")



if history:

    history_df = pd.DataFrame(history)

    print("\n  Training History (last 5 epochs):")

    print(history_df.tail(5).to_string(index=False))



experiment_summary = pd.DataFrame([

    {

        "model": "vilbert_v2",

        "best_val_f1": best_val_f1,

        "test_acc": test_metrics["acc"],

        "test_f1": test_metrics["f1"],

        "checkpoint": best_ckpt_path,

    }

])

print("\n  Experiment Summary:")

print(experiment_summary.to_string(index=False))



test_acc = test_metrics["acc"]

test_f1 = test_metrics["f1"]

all_preds = test_metrics["preds"]

all_labels = test_metrics["labels"]



print(f"\n{SEP}")

print(f"  Best checkpoint: {best_ckpt_path}")

print(SEP)


Loading best checkpoint from: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/checkpoints/vilbert_best.pt
Checkpoint epoch : 6   val_f1=0.6320   val_acc=0.7140

  Test Set Results
  Accuracy  (overall)   : 0.7456  (74.56%)
  Macro F1  (unweighted): 0.6543

  Per-Class Classification Report:
              precision    recall  f1-score   support

    negative     0.7167    0.6324    0.6719       136
     neutral     0.4884    0.4468    0.4667        47
    positive     0.7958    0.8550    0.8244       269

    accuracy                         0.7456       452
   macro avg     0.6670    0.6447    0.6543       452
weighted avg     0.7401    0.7456    0.7413       452

  Confusion Matrix  (rows=true, cols=pred):
            negative     neutral    positive
  negative          86           8          42
   neutral           9          21          17
  positive          25          14         230

  Training History (last 5 epochs):
 epoch  train_loss  train_acc  val_loss  val_acc   val_f1

In [12]:
# ─────────────────────────────────────────────

# Cell 11 — Save Model for Later Use

# ─────────────────────────────────────────────

SAVE_DIR = CONFIG["save_dir"]

os.makedirs(SAVE_DIR, exist_ok=True)



full_ckpt_path = os.path.join(SAVE_DIR, "vilbert_v2_full.pt")

weights_path = os.path.join(SAVE_DIR, "vilbert_v2_weights_only.pt")

config_path = os.path.join(SAVE_DIR, "vilbert_v2_config.json")

history_path = os.path.join(SAVE_DIR, "vilbert_v2_history.csv")

summary_path = os.path.join(SAVE_DIR, "vilbert_v2_summary.json")



torch.save(

    {

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "config": CONFIG,

        "label_map": LABEL_MAP,

        "label_imap": LABEL_IMAP,

        "best_val_f1": best_val_f1,

        "test_acc": test_acc,

        "test_f1": test_f1,

        "history": history,

    },

    full_ckpt_path,

)

print(f"Full checkpoint saved : {full_ckpt_path}")



torch.save(model.state_dict(), weights_path)

print(f"Weights-only saved    : {weights_path}")



config_to_save = {key: value for key, value in CONFIG.items() if not callable(value)}

config_to_save["label_map"] = LABEL_MAP

config_to_save["label_imap"] = {str(key): value for key, value in LABEL_IMAP.items()}

with open(config_path, "w") as file:

    json.dump(config_to_save, file, indent=2)

print(f"Config JSON saved     : {config_path}")



pd.DataFrame(history).to_csv(history_path, index=False)

print(f"History CSV saved     : {history_path}")



with open(summary_path, "w") as file:

    json.dump(

        {

            "model": "vilbert_v2",

            "best_val_f1": best_val_f1,

            "test_acc": test_acc,

            "test_f1": test_f1,

            "checkpoint": best_ckpt_path,

            "parquet_path": CONFIG["parquet_path"],

        },

        file,

        indent=2,

    )

print(f"Summary JSON saved    : {summary_path}")



for path in [full_ckpt_path, weights_path, config_path, history_path, summary_path]:

    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(f"  {os.path.basename(path):<35}  {size_mb:.2f} MB")



print("\n" + "=" * 60)

print("  HOW TO RELOAD THIS MODEL LATER")

print("=" * 60)

print("""

  import json, torch

  from transformers import BertTokenizer



  with open("vilbert_v2_config.json") as file:

      cfg = json.load(file)



  model = ViLBERT(cfg)

  checkpoint = torch.load("vilbert_v2_full.pt", map_location="cpu")

  model.load_state_dict(checkpoint["model_state_dict"])

  label_map = checkpoint["label_map"]

  label_imap = checkpoint["label_imap"]



  tokenizer = BertTokenizer.from_pretrained(cfg["bert_model_name"])

  model.eval()

  # logits = model(input_ids, attention_mask, token_type_ids, pixel_values)

  # pred = logits.argmax(dim=-1).item()

  # label = label_imap[pred]

""")


Full checkpoint saved : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final/vilbert_full.pt
Weights-only saved    : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final/vilbert_weights_only.pt
Config JSON saved     : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final/config.json
  vilbert_full.pt                      2087.5 MB
  vilbert_weights_only.pt              697.7 MB
  config.json                          0.0 MB

  HOW TO RELOAD THIS MODEL LATER

  import json, torch
  from transformers import BertModel, BertTokenizer

  # 1. Load config
  with open("config.json") as f:
      cfg = json.load(f)

  # 2. Rebuild model architecture
  model = ViLBERT(cfg)

  # 3a. Load full checkpoint (includes optimizer, history, metrics)
  ckpt = torch.load("vilbert_full.pt", map_location="cpu")
  model.load_state_dict(ckpt["model_state_dict"])
  label_map  = ckpt["label_map"]     # {"negative":0, "neutral":1, "positive":2}
  label_imap = ckpt["label_imap"]  